# Associate Ca2+ signal with position on cheeseboard for each session using crossregistration

Define Experiment type

In [ ]:
local = True
all_expe_types =['Cheeseboard']

if local:
    dir = "//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_data/L2_3_mice/"
else: 
    dir = "/crnldata/forgetting/Aurelie/MiniscopeOE_data/L2_3_mice/"

Load packages

In [ ]:
import os
import re
import numpy as np
from scipy import signal
import quantities as pq
import math 
import neo
import json
from collections import Counter
from pathlib import Path
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider, Button, Cursor
import pickle
import sys 
from datetime import datetime
import shutil
from ast import literal_eval
from scipy.signal import find_peaks
from scipy.stats import zscore
from scipy import stats
from itertools import groupby
from IPython.display import display
from scipy.interpolate import griddata
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
import matplotlib.pyplot as plt
from scipy import interpolate
import numpy.matlib
from sklearn.decomposition import PCA
from scipy import stats
import numpy as np
import pickle
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import resample
from scipy.signal import resample_poly
from math import gcd
import warnings
warnings.filterwarnings("ignore")
import sys
from scipy.interpolate import interp1d
from collections import defaultdict
import bisect
from scipy.ndimage import gaussian_filter
import random
from scipy.ndimage import gaussian_filter1d
from matplotlib.collections import LineCollection
from itertools import islice
from upsetplot import from_contents, UpSet
from matplotlib_venn import venn3

class Tee:
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

if local: 
    os.chdir("C:/Users/Manip2/SCRIPTS/minian/")
else: 
    sys.path.append("/home/aurelie.brecier/minian/")

    
minian_path = os.path.join(os.path.abspath('.'),'minian')
print("The folder used for minian procedures is : {}".format(minian_path))
sys.path.append(minian_path)

from minian.utilities import (
    TaskAnnotation,
    get_optimal_chk,
    load_videos,
    open_minian,
    save_minian,
)

Define parameters

In [ ]:
pixel_to_cm = 2.25  
table_center_x, table_center_y = 313, 283  # Center of the cheeseboard table on the video
table_center_x, table_center_y = 300, 270  # Center of the cheeseboard table on the video
table_center_x, table_center_y = 315, 275  # Center of the cheeseboard table on the video

table_radius = 290 / 2
square_size = pixel_to_cm * 6

sigma = 1  # Standard deviation for Gaussian kernel
max_row = int(table_radius*2/square_size)+1
max_col = int(table_radius*2/square_size)+1

bin_size = 10
bin_edges = np.arange(0, 361, bin_size)
bin_centers = np.radians(bin_edges[:-1] + bin_size/2)

Define functions

In [ ]:
def find_closest_index_sorted(arr, target):
    idx = bisect.bisect_left(arr, target)  # Find the insertion point
    if idx == 0:
        return 0
    if idx == len(arr):
        return len(arr) - 1
    before = idx - 1
    after = idx
    return before if abs(arr[before] - target) <= abs(arr[after] - target) else after

def calculate_relative_distance(x1, y1, x2, y2):
    return math.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

def find_long_non_nan_sequences(arr, min_length=100):
    mask = ~np.isnan(arr)  # True for non-NaN values
    diff = np.diff(np.concatenate(([0], mask.astype(int), [0])))  # Add padding to detect edges
    starts = np.where(diff == 1)[0]  # Where a sequence starts
    ends = np.where(diff == -1)[0]   # Where a sequence ends
    sequences = [arr[start:end] for start, end in zip(starts, ends) if (end - start) > min_length]
    return sequences

def replace_high_speed_points_with_nan(x, y, speed_threshold):
    x = np.array(x, dtype='float')
    y = np.array(y, dtype='float')
    # Compute speed between consecutive points
    dx = np.diff(x)
    dy = np.diff(y)
    speeds = np.sqrt(dx**2 + dy**2)
    # Create mask for speed exceeding threshold
    high_speed_mask = speeds > speed_threshold
    # We mark i+1 as NaN if speed between them is too high
    x_out = x.copy()
    y_out = y.copy()
    for i in range(len(high_speed_mask)):
        if high_speed_mask[i]:
            # Only mark the faster of the two points
            if i > 0 and i < len(x) - 1:
                if speeds[i] > speeds[i - 1]:
                    x_out[i + 1] = np.nan
                    y_out[i + 1] = np.nan
                else:
                    x_out[i] = np.nan
                    y_out[i] = np.nan
    return x_out, y_out

def interpolate_2d_path(x, y, kind='linear', fill='extrapolate'):
    x = np.array(x, dtype='float')
    y = np.array(y, dtype='float')
    indices = np.arange(len(x))
    valid_mask = ~np.isnan(x) & ~np.isnan(y)
    if np.sum(valid_mask) < 2:
        raise ValueError("Not enough valid points to interpolate/extrapolate.")
    interp_x = interp1d(indices[valid_mask], x[valid_mask], kind=kind, fill_value=fill, bounds_error=False)
    interp_y = interp1d(indices[valid_mask], y[valid_mask], kind=kind, fill_value=fill, bounds_error=False)
    x_filled = x.copy()
    y_filled = y.copy()
    nan_mask = np.isnan(x) | np.isnan(y)
    x_filled[nan_mask] = interp_x(indices[nan_mask])
    y_filled[nan_mask] = interp_y(indices[nan_mask])
    return x_filled, y_filled

def limit_speed(x, y, max_speed):
    dx = np.diff(x.copy())
    dy = np.diff(y.copy())
    speeds = np.sqrt(dx**2 + dy**2)
    for i,t in enumerate(speeds):
        if t > max_speed:        
            x[i+1] = x[i] 
            y[i+1] = y[i] 
            x[i+2] = x[i] 
            y[i+2] = y[i] 
    return x, y

def remove_short_sequences(arr, max_len=10):
    arr = np.array(arr, dtype='float')
    result = arr.copy()
    is_value = ~np.isnan(arr)
    i = 0
    while i < len(arr):
        if is_value[i]:
            start = i
            while i < len(arr) and is_value[i]:
                i += 1
            end = i
            seq_len = end - start
            # Check if surrounded by NaNs and short enough
            if seq_len <= max_len:
                left_nan = (start == 0) or np.isnan(arr[start - 1])
                right_nan = (end == len(arr)) or np.isnan(arr[end])  # safe for edge
                if left_nan and right_nan:
                    result[start:end] = np.nan
        else:
            i += 1
    return result

# Conversion des quaternions en angles d'Euler
def quaternion_to_euler(qw, qx, qy, qz):
    sinr_cosp = 2 * (qw * qx + qy * qz)
    cosr_cosp = 1 - 2 * (qx * qx + qy * qy)
    roll = np.arctan2(sinr_cosp, cosr_cosp)

    sinp = 2 * (qw * qy - qz * qx)
    sinp = np.clip(sinp, -1.0, 1.0)
    pitch = np.arcsin(sinp)

    siny_cosp = 2 * (qw * qz + qx * qy)
    cosy_cosp = 1 - 2 * (qy * qy + qz * qz)
    yaw = np.arctan2(siny_cosp, cosy_cosp)

    return roll, pitch, yaw

def direction(x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    if dx == 0 and dy == 0:
        return np.nan  # undefined direction
    angle_rad = math.atan2(dy, dx)
    angle_deg = math.degrees(angle_rad)
    return (angle_deg + 360) % 360

## Compute Real data

Create a Analysis folder

In [ ]:
FolderNameSave=str(datetime.now())[:19] # Get the current date and time
FolderNameSave = FolderNameSave.replace(" ", "_").replace(".", "_").replace(":", "_")
if local:
    destination_folder= f"//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/Exploration_task/0_NeuronIdentity_{FolderNameSave}" 
else: 
    destination_folder= f"/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/Exploration_task/0_NeuronIdentity_{FolderNameSave}" 
os.makedirs(destination_folder)
folder_to_save=Path(destination_folder)

Compute behavior and Ca2+ activity

In [ ]:
AllCellDict_PC={}
AllTimeDict_PC={}
AllCellDict_HD={}
AllTimeDict_HD={}
AllCellDict_Roll={}
AllTimeDict_Roll={}
AllCellDict_Pitch={}
AllTimeDict_Pitch={}
AllCellDict_Yaw={}
AllTimeDict_Yaw={}
AllCellDict_Speed={}
AllTimeDict_Speed={}
AllCellDict_AHV={}
AllTimeDict_AHV={}     

PlaceCell_infoSess={}
HeadDirection_infoSess={}
Roll_infoSess={}
Pitch_infoSess={}
Yaw_infoSess={}
AHV_infoSess={}
Speed_infoSess={}
                

for dpath in Path(dir).glob('**/Exploration_task/mappingsAB.pkl'):

    mappfile = open(dpath.parents[0]/ f'mappingsAB.pkl', 'rb')
    mapping = pickle.load(mappfile)
    mapping_sess = mapping['session']   
        
    centfile = open(dpath.parents[0]/ f'centsAB.pkl', 'rb')
    centroids = pickle.load(centfile) 

    mice = dpath.parents[1].parts[-1]
    NeuronType = dpath.parents[2].parts[-1]
    
    print(f"Processing mouse {mice} of type {NeuronType}")

    sess=0

    minian_folders = [f for f in dpath.parents[0].rglob('minian') if f.is_dir()]

    for minianpath in minian_folders: # for each minian folders found in this mouse 

        if any(p in all_expe_types for p in minianpath.parts): # have to be to the expe_types

            try: 
                session_path=minianpath.parents[1]        
                tsmini= pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])['Time Stamp (ms)']
                V4subfolder=False
                session_type=minianpath.parents[2].name
                session_date=minianpath.parents[3].name
                session_time=minianpath.parents[1].name
                session=session_time         
            except:
                session_path=minianpath.parents[2]      
                tsmini= pd.read_csv(list(session_path.glob('*V4_Miniscope/timeStamps.csv'))[0])['Time Stamp (ms)']                      
                V4subfolder=True
                session_type=minianpath.parents[4].name
                session_date=minianpath.parents[5].name
                session_time=minianpath.parents[0].name
                session=session_time
            print(f"Processing {session_type} session: {session} on the {session_date} ")

            # Minian data 
                        
            minian_ds = open_minian(minianpath)
            Co = minian_ds['C']  # calcium traces
            minian_freq=round(1/np.mean(np.diff(np.array(tsmini)/1000)))
            print('... miniscope sample rate =', minian_freq, 'Hz')            
            try: 
                TodropFile = minianpath / f'TodropFileAB.json'
                with open(TodropFile, 'r') as f:
                    unit_to_drop = json.load(f)
            except:
                TodropFile = minianpath.parent / f'TodropFileAB.json'
                with open(TodropFile, 'r') as f:
                    unit_to_drop = json.load(f)

            # Extraction des quaternions

            vestibular_df= pd.read_csv(list(session_path.glob('*V4_Miniscope/headOrientation.csv'))[0])
            tsvest = vestibular_df['Time Stamp (ms)']
            qw = vestibular_df['qw'].to_numpy()
            qx = vestibular_df['qx'].to_numpy()
            qy = vestibular_df['qy'].to_numpy()
            qz = vestibular_df['qz'].to_numpy()
            roll, pitch, yaw = quaternion_to_euler(qw, qx, qy, qz)
            roll_deg = np.degrees(roll) # nose goes clockwise or anticlockwise
            pitch_deg = np.degrees(pitch) # nose up or down
            yaw_deg = np.degrees(yaw) # nose moves from side to side   
            last_valid = tsvest.last_valid_index() + 1
            roll_deg = roll_deg[:last_valid]
            pitch_deg = pitch_deg[:last_valid]
            yaw_deg = yaw_deg[:last_valid]

            if V4subfolder:
                V4subfolder_id = int(minianpath.parent.name[-1]) - 1 
                ts_start = V4subfolder_id*15*1000 # cause 15 videos per subfolders and 1000 frames per videos
                ts_stop = np.shape(Co)[1] + ts_start 
                tsmini_sub=tsmini[ts_start:ts_stop]
                tsmini_sub=tsmini_sub.reset_index(drop=True)                  
                tsvest_sub = tsvest[ts_start:ts_stop]
                tsvest_sub = tsvest_sub.reset_index(drop=True)    
                roll_deg= roll_deg[ts_start:ts_stop]
                pitch_deg= pitch_deg[ts_start:ts_stop]
                yaw_deg= yaw_deg[ts_start:ts_stop]            
            else:
                tsmini_sub=tsmini  
                tsvest_sub=tsvest  
             
            # DeepLabCut data

            dlcfile=''
            dlcpath=Path(f'{Path(session_path)}/My_First_WebCam/')
            for file in os.listdir(dlcpath):
                if file.endswith(('.h5')):
                    dlcfile=file
                    break
            dlc_path = os.path.join(dlcpath, dlcfile)
            df = pd.read_hdf(dlc_path)
            directory = os.path.dirname(dlc_path)
            timestamps_path = Path(directory,'timeStamps.csv')
            if timestamps_path.exists():
                timestamps = pd.read_csv(timestamps_path)
                tswebcam = np.array(timestamps['Time Stamp (ms)'])
                frame_rate = round(1/(np.mean(np.diff(timestamps.iloc[:,1]))/1000))  # fps
            else:
                frame_rate = 16  # fps /!\ CHANGE ACCORDING TO YOUR DATA
            
            # Nose 

            df_nose= df.copy()
            df_nose.iloc[:, 0] = df_nose.apply(lambda row: row.iloc[0] if row.iloc[2] > 0.5 else np.nan, axis=1)
            df_nose.iloc[:, 1] = df_nose.apply(lambda row: row.iloc[1] if row.iloc[2] > 0.5 else np.nan, axis=1)
            X = df_nose.iloc[:, 0]
            Y = df_nose.iloc[:, 1]        
            nose_xO= np.array(X.values) 
            nose_yO = np.array(Y.values)
            for i, x in enumerate(nose_xO):# Define when the mouse is on the cheeseboard (start)
                y = nose_yO[i]
                if calculate_relative_distance(x, y, table_center_x, table_center_y) >= table_radius:
                    nose_xO[i] = np.nan
                    nose_yO[i] = np.nan
            nose_xOO = remove_short_sequences(nose_xO, max_len=3)
            nose_yOO = remove_short_sequences(nose_yO, max_len=3)
            x_start = find_long_non_nan_sequences(nose_xOO)[0][0] # first value of the first long non nan sequence
            y_start = find_long_non_nan_sequences(nose_yOO)[0][0] # first value of the first long non nan sequence
            start_frame = np.where(nose_xOO == x_start)[0][0].item()
            
            nose_xOO[:start_frame]=np.nan # remove any path before the real start
            nose_yOO[:start_frame]=np.nan # remove any path before the real start
            nose_x1, nose_y1 = replace_high_speed_points_with_nan(nose_xOO, nose_yOO, speed_threshold=10)
            last_frame = len(nose_x1)
            nose_x2, nose_y2 = interpolate_2d_path(nose_x1[start_frame:last_frame], nose_y1[start_frame:last_frame], kind='nearest')
            max_speed = 20
            nose_x3, nose_y3 = limit_speed(nose_x2, nose_y2, max_speed=max_speed)
            nose_x = np.concatenate((nose_x1[:start_frame], nose_x3))
            nose_y = np.concatenate((nose_y1[:start_frame], nose_y3))
            
            # Neck 

            df_neck= df.copy()
            df_neck.iloc[:, 3] = df_neck.apply(lambda row: row.iloc[3] if row.iloc[5] > 0.5 else np.nan, axis=1)
            df_neck.iloc[:, 4] = df_neck.apply(lambda row: row.iloc[4] if row.iloc[5] > 0.5 else np.nan, axis=1)
            Xt = df_neck.iloc[:, 3]
            Yt = df_neck.iloc[:, 4]        
            neck_xO= np.array(Xt.values) 
            neck_yO = np.array(Yt.values)
            for i, x in enumerate(neck_xO):# Define when the mouse is on the cheeseboard (start)
                y = neck_yO[i]
                if calculate_relative_distance(x, y, table_center_x, table_center_y) >= table_radius:
                    neck_xO[i] = np.nan
                    neck_yO[i] = np.nan
            neck_xOO = remove_short_sequences(neck_xO, max_len=3)
            neck_yOO = remove_short_sequences(neck_yO, max_len=3)
            neck_xOO[:start_frame]=np.nan # remove any path before the real start
            neck_yOO[:start_frame]=np.nan # remove any path before the real start
            neck_x1, neck_y1 = replace_high_speed_points_with_nan(neck_xOO, neck_yOO, speed_threshold=10)
            neck_x2, neck_y2 = interpolate_2d_path(neck_x1[start_frame:last_frame], neck_y1[start_frame:last_frame], kind='nearest')
            neck_x3, neck_y3 = limit_speed(neck_x2, neck_y2, max_speed=max_speed)
            neck_x = np.concatenate((neck_x1[:start_frame], neck_x3))
            neck_y = np.concatenate((neck_y1[:start_frame], neck_y3))
                        

            # Keep only crossregistered cells

            C_sel=Co.drop_sel(unit_id=unit_to_drop)
            indexMappList=mapping_sess[session]
            kept_uniq_unit_List=[]
            for unit in C_sel['unit_id'].values:
                indexMapp = np.where(indexMappList == unit)[0]
                kept_uniq_unit_List.append(str(indexMapp))
            nb_unit=len(C_sel['unit_id'].values)
            if nb_unit == 0:
                print(f'... no cells kept in the session: {session}')
                continue
            print(f'... {nb_unit} kept cells in the session')


            # If the miniscope recording started after the webcam recording, cut the webcam data
            if tsmini_sub.iloc[0] > tswebcam[start_frame]:
                Newstart_frame = np.where(tswebcam >= tsmini_sub.iloc[0].item())[0][1].item()
                print(f'... webcam data cut to match miniscope length, new start at frame {Newstart_frame} (instead of {start_frame})')
                start_frame = Newstart_frame 
            # If the miniscope recording is shorter than the webcam recording, cut the webcam data
            if tsmini_sub.iloc[-1] < tswebcam[last_frame-1]:
                Newlast_frame = np.where(tswebcam <= tsmini_sub.iloc[-1].item())[0][-1].item()
                print(f'... webcam data cut to match miniscope length, new end at frame {Newlast_frame} (instead of {last_frame})')
                last_frame = Newlast_frame
            

            # Align data

            nose_x_ = nose_x[start_frame:last_frame]
            nose_y_ = nose_y[start_frame:last_frame]
            neck_x_ = neck_x[start_frame:last_frame]
            neck_y_ = neck_y[start_frame:last_frame]            
            angles_deg = np.array([direction(x1, y1, x2, y2) for x1, y1, x2, y2 in zip(neck_x_, neck_y_, nose_x_, nose_y_)])

            ahv_deg= np.diff(angles_deg)
            ahv_deg_= np.concatenate(([0], ahv_deg))

            dx= np.diff(nose_x_)
            dy= np.diff(nose_y_)
            speed = np.sqrt(dx**2 + dy**2)
            speed_= np.concatenate(([0], speed))
            speed_= np.where(speed_>max_speed, 0, speed_)

            # Keep only crossregistered cells

            Carray = C_sel.to_numpy().T
            Calcium = pd.DataFrame(Carray.T, index=C_sel['unit_id'].values.tolist())            

            n = int(np.floor(2 * table_radius / square_size))
            for unit in range(nb_unit): 
                indexMapp = np.where(mapping_sess[session] == Calcium.index[unit])[0]
                if len(indexMapp)>0 : # The neuron needs to be in the cross-registration
                    unique_unit = f'{mice}{str(indexMapp[0])}'
                    sess_unit = f'{mice}{str(indexMapp[0])}_s{sess}_{unit}'
                    Carray_unit = Carray[:,unit]

                    nr_tot_act_biaised_PC = defaultdict(int) # Neuron activity per square
                    counts_PC = defaultdict(int) # Count visits per square

                    nr_tot_act_biaised_HD = np.zeros(len(bin_centers))
                    counts_HD = np.zeros(len(bin_centers)) 

                    nr_tot_act_biaised_Roll = np.zeros(len(bin_centers))
                    counts_Roll = np.zeros(len(bin_centers))
                    
                    nr_tot_act_biaised_Pitch = np.zeros(len(bin_centers))
                    counts_Pitch = np.zeros(len(bin_centers))           

                    nr_tot_act_biaised_Yaw = np.zeros(len(bin_centers))
                    counts_Yaw = np.zeros(len(bin_centers))
                    
                    nr_tot_act_biaised_Speed = np.zeros(max_speed)
                    counts_Speed = np.zeros(max_speed)

                    nr_tot_act_biaised_AHV = np.zeros(len(bin_centers)*2)
                    counts_AHV = np.zeros(len(bin_centers)*2)

                    for idx, (px, py, angle, _speed, _ahv) in enumerate(zip(nose_x_, nose_y_, angles_deg, speed_, ahv_deg_)):
                        if np.sqrt((px - table_center_x)**2 + (py - table_center_y)**2) > table_radius:
                            continue  # skip points outside circle
                        closest_point = find_closest_index_sorted(tsmini_sub, tswebcam[start_frame+idx])

                        ix = int(np.floor((px - (table_center_x - n/2 * square_size)) / square_size))
                        iy = int(np.floor((py - (table_center_y - n/2 * square_size)) / square_size))
                        nr_tot_act_biaised_PC[(ix, iy)] += Carray_unit[closest_point]
                        counts_PC[(ix, iy)] += 1/frame_rate 

                        bin_idx = int(angle // bin_size)
                        nr_tot_act_biaised_HD[bin_idx] += Carray_unit[closest_point]
                        counts_HD[bin_idx] += 1/frame_rate

                        speedbin_idx = min(int(_speed), 19) # between 0 and 20 
                        nr_tot_act_biaised_Speed[speedbin_idx] += Carray_unit[closest_point]
                        counts_Speed[speedbin_idx] += 1/frame_rate  

                        ahvbin_idx = int(_ahv // bin_size*2)
                        nr_tot_act_biaised_AHV[ahvbin_idx] += Carray_unit[closest_point]
                        counts_AHV[ahvbin_idx] += 1/frame_rate  

                    if len(roll_deg) != len(Carray_unit):
                        print('ISSSUE with length Calcium array & vestibular infos')

                    for idx, (_roll, _pitch, _yaw) in enumerate(zip(roll_deg, pitch_deg, yaw_deg)):

                        rollbin_idx = int(_roll // bin_size)
                        nr_tot_act_biaised_Roll[rollbin_idx] += Carray_unit[idx]
                        counts_Roll[rollbin_idx] += 1/minian_freq      

                        pitchbin_idx = int(_pitch // bin_size)
                        nr_tot_act_biaised_Pitch[pitchbin_idx] += Carray_unit[idx]
                        counts_Pitch[pitchbin_idx] += 1/minian_freq

                        yawbin_idx = int(_yaw // bin_size)
                        nr_tot_act_biaised_Yaw[yawbin_idx] += Carray_unit[idx]
                        counts_Yaw[yawbin_idx] += 1/minian_freq


                    if unique_unit in AllCellDict_PC:
                        AllCellDict_PC[unique_unit][session] = nr_tot_act_biaised_PC
                        AllTimeDict_PC[unique_unit][session] = counts_PC

                        AllCellDict_HD[unique_unit][session] = nr_tot_act_biaised_HD 
                        AllTimeDict_HD[unique_unit][session] = counts_HD

                        AllCellDict_Roll[unique_unit][session] = nr_tot_act_biaised_Roll 
                        AllTimeDict_Roll[unique_unit][session] = counts_Roll
                        
                        AllCellDict_Pitch[unique_unit][session] = nr_tot_act_biaised_Pitch 
                        AllTimeDict_Pitch[unique_unit][session] = counts_Pitch

                        AllCellDict_Yaw[unique_unit][session] = nr_tot_act_biaised_Yaw 
                        AllTimeDict_Yaw[unique_unit][session] = counts_Yaw

                        AllCellDict_AHV[unique_unit][session] = nr_tot_act_biaised_AHV 
                        AllTimeDict_AHV[unique_unit][session] = counts_AHV

                        AllCellDict_Speed[unique_unit][session] = nr_tot_act_biaised_Speed 
                        AllTimeDict_Speed[unique_unit][session] = counts_Speed

                    else :
                        AllCellDict_PC[unique_unit] = {}
                        AllCellDict_PC[unique_unit][session] = nr_tot_act_biaised_PC 
                        AllTimeDict_PC[unique_unit] = {}
                        AllTimeDict_PC[unique_unit][session] = counts_PC
                        
                        AllCellDict_HD[unique_unit] = {}
                        AllCellDict_HD[unique_unit][session] = nr_tot_act_biaised_HD 
                        AllTimeDict_HD[unique_unit] = {}
                        AllTimeDict_HD[unique_unit][session] = counts_HD

                        AllCellDict_Roll[unique_unit] = {}
                        AllCellDict_Roll[unique_unit][session] = nr_tot_act_biaised_Roll 
                        AllTimeDict_Roll[unique_unit] = {}
                        AllTimeDict_Roll[unique_unit][session] = counts_Roll

                        AllCellDict_Pitch[unique_unit] = {}
                        AllCellDict_Pitch[unique_unit][session] = nr_tot_act_biaised_Pitch 
                        AllTimeDict_Pitch[unique_unit] = {}
                        AllTimeDict_Pitch[unique_unit][session] = counts_Pitch

                        AllCellDict_Yaw[unique_unit] = {}
                        AllCellDict_Yaw[unique_unit][session] = nr_tot_act_biaised_Yaw 
                        AllTimeDict_Yaw[unique_unit] = {}
                        AllTimeDict_Yaw[unique_unit][session] = counts_Yaw

                        AllCellDict_AHV[unique_unit] = {}
                        AllCellDict_AHV[unique_unit][session] = nr_tot_act_biaised_AHV 
                        AllTimeDict_AHV[unique_unit] = {}
                        AllTimeDict_AHV[unique_unit][session] = counts_AHV

                        AllCellDict_Speed[unique_unit] = {}
                        AllCellDict_Speed[unique_unit][session] = nr_tot_act_biaised_Speed 
                        AllTimeDict_Speed[unique_unit] = {}
                        AllTimeDict_Speed[unique_unit][session] = counts_Speed

                    ###### Each session is a key ######

                    nr_tot_act_array = [[None for _ in range(max_col)] for _ in range(max_row)] # Initialize 2D array
                    for (row, col), value in nr_tot_act_biaised_PC.items():# Fill array
                        nr_tot_act_array[row][col] = value    
                    nr_tot_act_array = [[np.nan if v is None else v for v in row] for row in nr_tot_act_array] # Replace None with np.nan
                    nr_tot_act_array = np.array(nr_tot_act_array, dtype=float)
                    nr_tot_act_array=nr_tot_act_array.T
                    array_for_filter = np.nan_to_num(nr_tot_act_array, nan=0.0) # Replace nan with 0 for Gaussian filtering
                    act_smoothed_array = gaussian_filter(array_for_filter, sigma=sigma)    # Apply 2D Gaussian filter

                    nr_tot_time_array = [[None for _ in range(max_col)] for _ in range(max_row)] # Initialize 2D array
                    for (row, col), value in counts_PC.items():# Fill array
                        nr_tot_time_array[row][col] = value
                    nr_tot_time_array = [[np.nan if v is None else v for v in row] for row in nr_tot_time_array] # Replace None with np.nan
                    nr_tot_time_array = np.array(nr_tot_time_array, dtype=float)
                    nr_tot_time_array=nr_tot_time_array.T
                    array_for_filter = np.nan_to_num(nr_tot_time_array, nan=0.0) # Replace nan with 0 for Gaussian filtering
                    time_smoothed_array = gaussian_filter(array_for_filter, sigma=sigma)    # Apply 2D Gaussian filter
                    time_smoothed_array[time_smoothed_array < 0.1] = np.nan  # do not consider if spent less than 0.1 second in the square

                    # Create a circular mask
                    rows, cols = act_smoothed_array.shape
                    center_row, center_col = rows // 2, cols // 2
                    radius = min(rows, cols) // 2   
                    Y, X = np.ogrid[:rows, :cols]
                    dist_from_center = np.sqrt((X - center_col)**2 + (Y - center_row)**2)
                    mask = dist_from_center < radius  
                    act_smoothed_array_mask = np.where(mask, act_smoothed_array, np.nan)# Apply mask: set values outside circle to NaN
                    time_smoothed_array_mask = np.where(mask, time_smoothed_array, np.nan)# Apply mask: set values outside circle to NaN
                    PlaceCell_infoSess[sess_unit] = np.divide(act_smoothed_array_mask, time_smoothed_array_mask)

                    HeadDirection_infoSess[sess_unit] = np.divide(nr_tot_act_biaised_HD, np.where(counts_HD < 0.1, np.nan, counts_HD))  
                    Roll_infoSess[sess_unit] = np.divide(nr_tot_act_biaised_Roll, np.where(counts_Roll < 0.1, np.nan, counts_Roll))  
                    Pitch_infoSess[sess_unit] = np.divide(nr_tot_act_biaised_Pitch, np.where(counts_Pitch < 0.1, np.nan, counts_Pitch))  
                    Yaw_infoSess[sess_unit] = np.divide(nr_tot_act_biaised_Yaw, np.where(counts_Yaw < 0.1, np.nan, counts_Yaw))  
                    AHV_infoSess[sess_unit] = np.divide(nr_tot_act_biaised_AHV, np.where(counts_AHV < 0.1, np.nan, counts_AHV))  
                    Speed_infoSess[sess_unit] = np.divide(nr_tot_act_biaised_Speed, np.where(counts_Speed < 0.1, np.nan, counts_Speed))  
                    
            sess+=1

    print(f'{len(AllCellDict_PC.keys())} unique cells found so far')  
    
#########################################################################################
                # Average activity map across sessions for each cell #
##########################################################################################

# Compute Place cells activity
Spatial_info={}
PlaceCell_info={}
for uniquecell in AllCellDict_PC.keys():
    sums = {}
    for subdict in AllCellDict_PC[uniquecell].values():
        for key, value in subdict.items():
            sums[key] = sums.get(key, 0) + value
    nr_tot_act =  {key: sums[key] for key in sums}
    nr_tot_act_array = [[None for _ in range(max_col)] for _ in range(max_row)] # Initialize 2D array
    for (row, col), value in nr_tot_act.items():# Fill array
        nr_tot_act_array[row][col] = value    
    nr_tot_act_array = [[np.nan if v is None else v for v in row] for row in nr_tot_act_array] # Replace None with np.nan
    nr_tot_act_array = np.array(nr_tot_act_array, dtype=float)
    nr_tot_act_array=nr_tot_act_array.T

    array_for_filter = np.nan_to_num(nr_tot_act_array, nan=0.0) # Replace nan with 0 for Gaussian filtering
    act_smoothed_array = gaussian_filter(array_for_filter, sigma=sigma)    # Apply 2D Gaussian filter

    sums = {}
    for subdict in AllTimeDict_PC[uniquecell].values():
        for key, value in subdict.items():
            sums[key] = sums.get(key, 0) + value
    nr_tot_time = {key: sums[key] for key in sums}
    nr_tot_time_array = [[None for _ in range(max_col)] for _ in range(max_row)] # Initialize 2D array
    for (row, col), value in nr_tot_time.items():# Fill array
        nr_tot_time_array[row][col] = value
    nr_tot_time_array = [[np.nan if v is None else v for v in row] for row in nr_tot_time_array] # Replace None with np.nan
    nr_tot_time_array = np.array(nr_tot_time_array, dtype=float)
    nr_tot_time_array=nr_tot_time_array.T

    array_for_filter = np.nan_to_num(nr_tot_time_array, nan=0.0) # Replace nan with 0 for Gaussian filtering
    time_smoothed_array = gaussian_filter(array_for_filter, sigma=sigma)    # Apply 2D Gaussian filter
    time_smoothed_array[time_smoothed_array < .5] = np.nan  # do not consider if spent less than 0.1 second in the square

    # --- Mask outside circle ---
    rows, cols = max_row, max_col
    center_row, center_col = rows // 2, cols // 2
    radius = min(rows, cols) // 2   
    Y, X = np.ogrid[:rows, :cols]# Create a circular mask
    dist_from_center = np.sqrt((X - center_col)**2 + (Y - center_row)**2)
    mask = dist_from_center < radius    
    act_smoothed_array_mask = np.where(mask, act_smoothed_array, np.nan)# Apply mask: set values outside circle to NaN
    time_smoothed_array_mask = np.where(mask, time_smoothed_array, np.nan)# Apply mask: set values outside circle to NaN

    PlaceCell_info[uniquecell] = np.divide(act_smoothed_array_mask, time_smoothed_array_mask)
    meanact_smoothed_array_mask = PlaceCell_info[uniquecell]

    # --- Spatial information ---
    mean_act=np.nanmean(meanact_smoothed_array_mask)
    sum_time=np.nansum(time_smoothed_array_mask)
    Spatial_info[uniquecell] = 0
    for rows in np.arange(np.shape(meanact_smoothed_array_mask)[0]) : 
        for cols in np.arange(np.shape(meanact_smoothed_array_mask)[1]) :             
            bin_act=meanact_smoothed_array_mask[rows,cols]
            p_bin=time_smoothed_array_mask[rows,cols]/sum_time 
            if ~ np.isnan(bin_act) and ~ np.isnan(p_bin): 
                if bin_act!=0:             
                    spatial_info_bin=p_bin*(bin_act/mean_act)*math.log2(bin_act/mean_act)
                else:
                    spatial_info_bin=0    
                Spatial_info[uniquecell] += spatial_info_bin

# Compute HD activity
HeadDirection_info={}
for uniquecell, inner_dict in AllCellDict_HD.items():
    first_array = next(iter(inner_dict.values()))
    sumact_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        sumact_array += arr
    inner_dict= AllTimeDict_HD[uniquecell]
    timesum_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        timesum_array += arr
    timesum_array = np.where(timesum_array < 0.5, np.nan, timesum_array)
    
    HeadDirection_info[uniquecell] = np.divide(sumact_array,timesum_array)    

# Compute Roll activity
Roll_info={}
for uniquecell, inner_dict in AllCellDict_Roll.items():
    first_array = next(iter(inner_dict.values()))
    sumact_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        sumact_array += arr
    inner_dict= AllTimeDict_Roll[uniquecell]
    timesum_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        timesum_array += arr
    timesum_array = np.where(timesum_array < 0.5, np.nan, timesum_array)
    
    Roll_info[uniquecell] = np.divide(sumact_array,timesum_array)    

# Compute Pitch activity
Pitch_info={}
for uniquecell, inner_dict in AllCellDict_Pitch.items():
    first_array = next(iter(inner_dict.values()))
    sumact_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        sumact_array += arr
    inner_dict= AllTimeDict_Pitch[uniquecell]
    timesum_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        timesum_array += arr
    timesum_array = np.where(timesum_array < 0.5, np.nan, timesum_array)
    
    Pitch_info[uniquecell] = np.divide(sumact_array,timesum_array)    

# Compute Yaw activity
Yaw_info={}
for uniquecell, inner_dict in AllCellDict_Yaw.items():
    first_array = next(iter(inner_dict.values()))
    sumact_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        sumact_array += arr
    inner_dict= AllTimeDict_Yaw[uniquecell]
    timesum_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        timesum_array += arr
    timesum_array = np.where(timesum_array < 0.5, np.nan, timesum_array)
    
    Yaw_info[uniquecell] = np.divide(sumact_array,timesum_array)    

# Compute AHV activity
AHV_info={}
for uniquecell, inner_dict in AllCellDict_AHV.items():
    first_array = next(iter(inner_dict.values()))
    sumact_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        sumact_array += arr
    inner_dict= AllTimeDict_AHV[uniquecell]
    timesum_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        timesum_array += arr
    timesum_array = np.where(timesum_array < 0.5, np.nan, timesum_array)
   
    AHV_info[uniquecell] = np.divide(sumact_array,timesum_array)    

# Compute Speed activity
Speed_info={}
for uniquecell, inner_dict in AllCellDict_Speed.items():
    first_array = next(iter(inner_dict.values()))
    sumact_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        sumact_array += arr
    inner_dict= AllTimeDict_Speed[uniquecell]
    timesum_array = np.zeros_like(first_array)
    for arr in inner_dict.values():
        timesum_array += arr
    timesum_array = np.where(timesum_array < 0.5, np.nan, timesum_array)

    Speed_info[uniquecell] = np.divide(sumact_array,timesum_array)    

print(f'Total unique cell = {len(Speed_info.keys())}')  

Save dictionaries

In [ ]:
with open(os.path.join(folder_to_save, f"PlaceCellInfo.pkl"), "wb") as f:
    pickle.dump(PlaceCell_info, f)
with open(os.path.join(folder_to_save, f"PlaceCellInfoSess.pkl"), "wb") as f:
    pickle.dump(PlaceCell_infoSess, f)
with open(os.path.join(folder_to_save, f"SpatialInfo.pkl"), "wb") as f:
    pickle.dump(Spatial_info, f)
    
with open(os.path.join(folder_to_save, f"HeadDirectionInfo.pkl"), "wb") as f:
    pickle.dump(HeadDirection_info, f)
with open(os.path.join(folder_to_save, f"HeadDirectionInfoSess.pkl"), "wb") as f:
    pickle.dump(HeadDirection_infoSess, f)

with open(os.path.join(folder_to_save, f"AHVInfo.pkl"), "wb") as f:
    pickle.dump(AHV_info, f)
with open(os.path.join(folder_to_save, f"AHVInfoSess.pkl"), "wb") as f:
    pickle.dump(AHV_infoSess, f)

with open(os.path.join(folder_to_save, f"SpeedInfo.pkl"), "wb") as f:
    pickle.dump(Speed_info, f)
with open(os.path.join(folder_to_save, f"SpeedInfoSess.pkl"), "wb") as f:
    pickle.dump(Speed_infoSess, f)

with open(os.path.join(folder_to_save, f"RollInfo.pkl"), "wb") as f:
    pickle.dump(Roll_info, f)
with open(os.path.join(folder_to_save, f"RollInfoSess.pkl"), "wb") as f:
    pickle.dump(Roll_infoSess, f)

with open(os.path.join(folder_to_save, f"PitchInfo.pkl"), "wb") as f:
    pickle.dump(Pitch_info, f)
with open(os.path.join(folder_to_save, f"PitchInfoSess.pkl"), "wb") as f:
    pickle.dump(Pitch_infoSess, f)

with open(os.path.join(folder_to_save, f"YawInfo.pkl"), "wb") as f:
    pickle.dump(Yaw_info, f)
with open(os.path.join(folder_to_save, f"YawInfoSess.pkl"), "wb") as f:
    pickle.dump(Yaw_infoSess, f)

## Load Real & Schuffled Data

In [ ]:
folder_to_save= Path("//10.69.168.1/crnldata/forgetting/Aurelie/MiniscopeOE_analysis/Exploration_task/0_NeuronIdentity_2025-11-19_18_17_07_newtable/")

Load Shuffled Infos (permutation on circularly shifted calcium traces, done on the cluster)

In [ ]:
with open(f'{folder_to_save}/ShuffledSpatialInfo.pkl', "rb") as f:
    ShuffledSpatial_info = pickle.load(f)
with open(f'{folder_to_save}/ShuffledHeadDirectionInfo.pkl', "rb") as f:
    ShuffledHeadDirection_info = pickle.load(f)
with open(f'{folder_to_save}/ShuffledRollInfo.pkl', "rb") as f:
    ShuffledRoll_info = pickle.load(f)
with open(f'{folder_to_save}/ShuffledPitchInfo.pkl', "rb") as f:
    ShuffledPitch_info = pickle.load(f)
with open(f'{folder_to_save}/ShuffledYawInfo.pkl', "rb") as f:
    ShuffledYaw_info = pickle.load(f)
with open(f'{folder_to_save}/ShuffledSpeedInfo.pkl', "rb") as f:
    ShuffledSpeed_info = pickle.load(f)
with open(f'{folder_to_save}/ShuffledAHVInfo.pkl', "rb") as f:
    ShuffledAHV_info = pickle.load(f)

Load Infos (if necessary)

In [ ]:
with open(f'{folder_to_save}/SpatialInfo.pkl', "rb") as f:
    Spatial_info = pickle.load(f)
with open(f'{folder_to_save}/PlaceCellInfo.pkl', "rb") as f:
    PlaceCell_info = pickle.load(f)
with open(f'{folder_to_save}/HeadDirectionInfo.pkl', "rb") as f:
    HeadDirection_info = pickle.load(f)
with open(f'{folder_to_save}/RollInfo.pkl', "rb") as f:
    Roll_info = pickle.load(f)
with open(f'{folder_to_save}/PitchInfo.pkl', "rb") as f:
    Pitch_info = pickle.load(f)
with open(f'{folder_to_save}/YawInfo.pkl', "rb") as f:
    Yaw_info = pickle.load(f)
with open(f'{folder_to_save}/SpeedInfo.pkl', "rb") as f:
    Speed_info = pickle.load(f)
with open(f'{folder_to_save}/AHVInfo.pkl', "rb") as f:
    AHV_info = pickle.load(f)

with open(f'{folder_to_save}/PlaceCellInfoSess.pkl', "rb") as f:
    PlaceCell_infoSess = pickle.load(f)
with open(f'{folder_to_save}/HeadDirectionInfoSess.pkl', "rb") as f:
    HeadDirection_infoSess = pickle.load(f)
with open(f'{folder_to_save}/RollInfoSess.pkl', "rb") as f:
    Roll_infoSess = pickle.load(f)
with open(f'{folder_to_save}/PitchInfoSess.pkl', "rb") as f:
    Pitch_infoSess = pickle.load(f)
with open(f'{folder_to_save}/YawInfoSess.pkl', "rb") as f:
    Yaw_infoSess = pickle.load(f)
with open(f'{folder_to_save}/SpeedInfoSess.pkl', "rb") as f:
    Speed_infoSess = pickle.load(f)
with open(f'{folder_to_save}/AHVInfoSess.pkl', "rb") as f:
    AHV_infoSess = pickle.load(f)

Display results of permutations & save list of Neuron ID

In [ ]:
sorted_keys = sorted(ShuffledSpatial_info.keys())
ShuffledSpatial_info = {key: ShuffledSpatial_info[key] for rank, key in enumerate(sorted_keys)}
nb_tot_permutation = int(np.mean([len(v) for k, v in ShuffledSpatial_info.items()]))
print(nb_tot_permutation, 'permutations performed')

AllCellList=[]   

PlaceCellList=[]
HeadDirectionCellList=[]    
AHVCellList=[]    
HeadMouvementCellList=[]  
SpeedCellList=[]    
YawCellList=[]
RollCellList=[]
PitchCellList=[]

MouvementCellList= []  
SpatialCellList=[]
UndescribedCellList=[]
NonSpatialCellList=[]

PCCellList=[]
HDCellList=[]
PCHDCellList=[]
BodyMouvementCellList=[]

for nr in ShuffledSpatial_info.keys():
    count_higher=0
    AllCellList.append(nr)
    for permutation_nb in range(nb_tot_permutation):
        if ShuffledSpatial_info[nr][permutation_nb]>=Spatial_info[nr]:
            count_higher+=1
    p_value=(count_higher+1)/(nb_tot_permutation+1) # +1 for the observed value, +1 for the total number of shuffling
    
    PlaceCell = True if p_value<0.05 else False
    
    alpha = 0.01
    nb_bin_sign = 1
    observed_psth = HeadDirection_info[nr]
    shuffled_psths = arr = list(ShuffledHeadDirection_info[nr].values())
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_binsHD =  (observed_psth > upper_bound) # | (observed_psth < lower_bound) |

    HeadDirection = True if sum(significant_binsHD)>=nb_bin_sign else False

    observed_psth = AHV_info[nr]
    shuffled_psths = arr = list(ShuffledAHV_info[nr].values())
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_binsAHV =  (observed_psth > upper_bound) # | (observed_psth < lower_bound) |

    AHV = True if sum(significant_binsAHV)>=nb_bin_sign else False

    observed_psth = Roll_info[nr]
    shuffled_psths = arr = list(ShuffledRoll_info[nr].values())
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_binsRoll =  (observed_psth > upper_bound)

    observed_psth = Pitch_info[nr]
    shuffled_psths = arr = list(ShuffledPitch_info[nr].values())
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_binsPitch =  (observed_psth > upper_bound)

    observed_psth = Yaw_info[nr]
    shuffled_psths = arr = list(ShuffledYaw_info[nr].values())
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_binsYaw =  (observed_psth > upper_bound)

    Yaw = True if sum(significant_binsYaw)>=nb_bin_sign else False
    Pitch = True if sum(significant_binsPitch)>=nb_bin_sign else False
    Roll = True if sum(significant_binsRoll)>=nb_bin_sign else False

    HeadMouvement = True if sum(significant_binsYaw)>=nb_bin_sign or sum(significant_binsPitch)>=nb_bin_sign or sum(significant_binsRoll)>=nb_bin_sign else False

    observed_psth = Speed_info[nr]
    shuffled_psths = arr = list(ShuffledSpeed_info[nr].values())
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_binsSpeed =  (observed_psth > upper_bound)

    Speed = True if sum(significant_binsSpeed)>=nb_bin_sign else False

    
    if HeadMouvement :
        HeadMouvementCellList.append(nr)
    if Speed :
        SpeedCellList.append(nr)   
    if AHV:
        AHVCellList.append(nr)     
    if HeadDirection :
        HeadDirectionCellList.append(nr)
    if PlaceCell :
        PlaceCellList.append(nr)
    if Yaw :
        YawCellList.append(nr)
    if Pitch :
        PitchCellList.append(nr)
    if Roll :
        RollCellList.append(nr)


    if (HeadMouvement or Speed or AHV):
        MouvementCellList.append(nr)
    if (HeadDirection or PlaceCell) :
        SpatialCellList.append(nr)  
    if not(HeadDirection or PlaceCell or HeadMouvement or Speed or AHV):
        UndescribedCellList.append(nr)
    if not(HeadDirection or PlaceCell):
        NonSpatialCellList.append(nr)

    if (HeadMouvement or Speed or AHV) and not(HeadDirection or PlaceCell):
        BodyMouvementCellList.append(nr)
    if (HeadDirection) and not(PlaceCell):
        HDCellList.append(nr)
    if (PlaceCell) and not(HeadDirection):
        PCCellList.append(nr)
    if (PlaceCell) and (HeadDirection):
        PCHDCellList.append(nr)


plt.close('all')

# Spatial vs Body Venn diagram 
set1 = set(PlaceCellList)
set2 = set(HeadDirectionCellList)
set3 = set(MouvementCellList)
venn3([set1, set2, set3], set_colors=["#00CCFF", "#001AFF", "#FFAE00"], alpha=.6,  set_labels=("Place cell", "Head Direction cell", "Body-motion cell"))
plt.savefig(f"{folder_to_save}/PCHDMotion_venndistrib.svg", format="svg", dpi=300, transparent=True)
plt.show()

# All parameters UpSet diagram
categories = {
    "Place": PlaceCellList,
    "HD": HeadDirectionCellList,
    "AHV": AHVCellList,
    "Roll": RollCellList,
    "Yaw": YawCellList,
    "Pitch": PitchCellList,
    "Speed": SpeedCellList,
}
data = from_contents(categories)
upset = UpSet(data, subset_size='count', show_counts=True, min_subset_size = '1%')
upset.plot()
plt.savefig(f"{folder_to_save}/AllID_upsetdistrib.svg", format="svg", dpi=300, transparent=True)
plt.show()

In [ ]:
List={}
List['all']=AllCellList
List['Spatial']=SpatialCellList
List['NonSpatial']=BodyMouvementCellList

List['PC']=PCCellList
List['HD']=HDCellList
List['PC&HD']=PCHDCellList
List['BodyMouvement']=BodyMouvementCellList

with open(os.path.join(folder_to_save, f"List_SignCells.pkl"), "wb") as f:
    pickle.dump(List, f)

def clean_count(lst):# Helper function to clean names and count
    return Counter(re.sub(r'\d+', '', name) for name in lst)

# Count occurrences
counts1   = clean_count(PCCellList)
counts2  = clean_count(HDCellList)
counts3   = clean_count(PCHDCellList)
counts4   = clean_count(BodyMouvementCellList)
countsAll = clean_count(AllCellList)

# Common keys and corresponding values
keys = sorted(set(counts1)| set(counts2)|set(counts3)|set(counts4)| set(countsAll))
A, B, C, D, E = (np.array([c.get(k, 0) for k in keys]) for c in (counts1, counts2,counts3, counts4, countsAll))

# --- Create figure with 2 subplots side by side ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(5, 3), sharey=False)

# --- Plot 1: Raw counts ---
ax1.bar(keys, A, label='Place cells', color='cyan')
ax1.bar(keys, B, bottom=A, label='HD cells', color='blue')
ax1.bar(keys, C, bottom=B+A, label='PC&HD cells', color='purple')
ax1.bar(keys, D, bottom=C+B+A, label='Body motion', color='orange')
ax1.set_ylabel('Count')
ax1.set_title('Cell counts per mouse')
ax1.legend(fontsize=8)

# --- Plot 2: Percentages ---
pctA, pctB, pctC, pctD= (100 * X / E for X in (A, B, C, D))
ax2.bar(keys, pctA, label='Place cells', color='cyan')
ax2.bar(keys, pctB, bottom=pctA, label='HD cells', color='blue')
ax2.bar(keys, pctC, bottom=pctB+pctA, label='PC&HD cells', color='purple')
ax2.bar(keys, pctD, bottom=pctB+pctA+pctC, label='Body motion', color='orange')
ax2.set_ylabel('Percentage (%)')
ax2.set_title('Cell percentages per mouse')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{folder_to_save}/cell_distributions.svg", format="svg", dpi=300, transparent=True)
plt.show()

# Plot examples

Select a subset of cell to plot

In [ ]:
selected_keys = random.sample(list(Spatial_info.keys()), 64)
rand_id= random.randint(0, 1000)

selected_keys = list(set(HeadDirectionCellList) & set(YawCellList) - (set(PlaceCellList) | set(RollCellList) | set(PitchCellList) | set(AHVCellList)| set(SpeedCellList)))
rand_id ='HeadDirection&YawCellList'
print('Extract n°', rand_id)

Spatial_info_extract = {k: Spatial_info[k] for k in selected_keys}
PlaceCell_info_extract = {k: PlaceCell_info[k] for k in selected_keys}
HeadDirection_info_extract = {k: HeadDirection_info[k] for k in selected_keys}
Roll_info_extract = {k: Roll_info[k] for k in selected_keys}
Pitch_info_extract = {k: Pitch_info[k] for k in selected_keys}
Yaw_info_extract = {k: Yaw_info[k] for k in selected_keys}
Speed_info_extract = {k: Speed_info[k] for k in selected_keys}
AHV_info_extract = {k: AHV_info[k] for k in selected_keys}

selected_keys_ = [s + "_" for s in selected_keys]
PlaceCell_infoSess_extract = {k: v for k, v in PlaceCell_infoSess.items()if any(k.startswith(p) for p in selected_keys_)}
HeadDirection_infoSess_extract = {k: v for k, v in HeadDirection_infoSess.items()if any(k.startswith(p) for p in selected_keys_)}
Roll_infoSess_extract = {k: v for k, v in Roll_infoSess.items()if any(k.startswith(p) for p in selected_keys_)}
Pitch_infoSess_extract = {k: v for k, v in Pitch_infoSess.items()if any(k.startswith(p) for p in selected_keys_)}
Yaw_infoSess_extract = {k: v for k, v in Yaw_infoSess.items()if any(k.startswith(p) for p in selected_keys_)}
AHV_infoSess_extract = {k: v for k, v in AHV_infoSess.items()if any(k.startswith(p) for p in selected_keys_)}
Speed_infoSess_extract = {k: v for k, v in Speed_infoSess.items()if any(k.startswith(p) for p in selected_keys_)}

In [ ]:
save = 0

Map of Place fields

In [ ]:
sorted_keys = sorted(PlaceCell_info_extract.keys())
PlaceCell_info_extract = {key: PlaceCell_info_extract[key] for rank, key in enumerate(sorted_keys)}

nb_subplot=len(PlaceCell_info_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

fig, axs = plt.subplots(rows, cols, figsize=(8, 8))
axs = axs.flatten()

plt.tight_layout()

mean_ShuffledSpatial_info = {key: np.mean(list(subdict.values())).round(1)
    for key, subdict in ShuffledSpatial_info.items()}

for nsubplot, nr in enumerate(PlaceCell_info_extract.keys()):

    # Mask outside circle 
    rows, cols = np.full((max_row, max_col),1).shape
    center_row, center_col = rows // 2, cols // 2
    radius = min(rows, cols) // 2   
    Y, X = np.ogrid[:rows, :cols]# Create a circular mask
    dist_from_center = np.sqrt((X - center_col)**2 + (Y - center_row)**2)
    mask = dist_from_center < radius    
    masked_array = np.where(mask, np.full((max_row, max_col),1), np.nan)# Apply mask: set values outside circle to NaN
    
    # Draw map
    axs[nsubplot].imshow(masked_array, cmap='Greys_r', origin='upper')
    axs[nsubplot].imshow(PlaceCell_info_extract[nr], cmap='jet', origin='upper')
    axs[nsubplot].set_aspect('equal')
    clr= 'black' if nr in PlaceCellList else 'grey'
    ftw = 'bold' if nr in PlaceCellList else 'normal'
    axs[nsubplot].set_title(f'{nr} ({np.round(Spatial_info[nr], 1)}/{mean_ShuffledSpatial_info[nr]})', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)
        
    # Remove box (spines)
    for spine in axs[nsubplot].spines.values():
        spine.set_visible(False)
    axs[nsubplot].set_xticks([])  # No x-axis ticks
    axs[nsubplot].set_yticks([])  # No y-axis ticks

# Adjust layout to avoid clipping
fig.subplots_adjust(wspace=.1, hspace=.2)
plt.savefig(f'{folder_to_save}/PlaceFields_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()

Polar plots of HD 

In [ ]:
sorted_keys = sorted(HeadDirection_info_extract.keys())
HeadDirection_info_extract = {key: HeadDirection_info_extract[key] for rank, key in enumerate(sorted_keys)}

nb_subplot=len(HeadDirection_info_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

# Create the figure with a 2x2 grid
fig, axs = plt.subplots(rows, cols, figsize=(8, 8),  subplot_kw={'projection': 'polar'})
axs = axs.flatten()
plt.tight_layout()

for nsubplot, nr in enumerate(HeadDirection_info_extract.keys()):

    ax = axs[nsubplot]
    ax.set_theta_zero_location("E")  # 0° at East
    ax.set_theta_direction(-1)        # Clockwise
    
    clr= 'black' if nr in HeadDirectionCellList else 'grey'
    ftw = 'bold' if nr in HeadDirectionCellList else 'normal'
    ax.set_title(f'{nr}', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)
    
    theta = bin_centers
    theta = np.append(theta, theta[0])  
    r_smooth = gaussian_filter1d(HeadDirection_info_extract[nr], sigma=1, mode='wrap')  # wrap for circular smoothing
    r_smooth = np.append(r_smooth, r_smooth[0])
    pref_idx = np.argmax(r_smooth)    # Find preferred angle
    pref_angle = theta[pref_idx]    
    color = plt.cm.hsv(pref_angle / (2*np.pi))# pref_angle ranges 0 to 2*pi -> normalize to 0-1 for colormap
    ax.plot(theta, r_smooth, color=color)

    r_shuffled = np.mean(list(ShuffledHeadDirection_info[nr].values()),0)
    r_smooth = gaussian_filter1d(r_shuffled, sigma=1, mode='wrap')  # wrap for circular smoothing
    r_smooth = np.append(r_smooth, r_smooth[0])
    ax.plot(theta, r_smooth, color='grey')
    
    ax.set_xticks([])
    ax.set_yticks([])

fig.subplots_adjust(wspace=.1, hspace=.1)
plt.savefig(f'{folder_to_save}/HeadDirection_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()


Hist plots of Yaw

In [ ]:
sorted_keys = sorted(Yaw_info_extract.keys())
Yaw_info_extract = {key: Yaw_info_extract[key] for rank, key in enumerate(sorted_keys)}


nb_subplot=len(Yaw_info_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

plt.close('all')
fig,axs = plt.subplots(rows, cols, figsize=(8, 8))
axs = axs.flatten()
plt.tight_layout()

for nsubplot, nr in enumerate(Yaw_info_extract.keys()):

    # Compute the observed PSTH
    observed_psth = Yaw_info_extract[nr]
    shuffled_psths = arr = list(ShuffledYaw_info[nr].values())
    time_bins = np.linspace(-180, 180, int(360//bin_size))
    shift = int(len(time_bins)/2)  
    observed_psth = np.roll(observed_psth, shift)
    shuffled_psths = np.roll(shuffled_psths, shift)

    alpha = 0.01
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_bins01 =  (observed_psth > upper_bound)

    ax = axs[nsubplot]
    
    ax.axvline(x=0, linestyle='--', color='black', alpha=0.5)
    #ax.bar(time_bins, shuffled_psths[0], color='gray', width=bin_size, alpha=0.5)
    ax.bar(time_bins, np.mean(shuffled_psths, axis=0), color='gray', width=bin_size, alpha=0.5)
    ax.bar(time_bins, observed_psth, color='red', width=bin_size, alpha=0.5)

    ax.scatter(
        time_bins[significant_bins01],
        observed_psth[significant_bins01],
        color='black',
        label='p<0.01', s=10, marker = '*',
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim([-180, 180])
    clr= 'black' if nr in YawCellList else 'grey'
    ftw = 'bold' if nr in YawCellList else 'normal'
    ax.set_title(f'{nr}', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)

fig.subplots_adjust(wspace=.1, hspace=.2)
plt.savefig(f'{folder_to_save}/Yaw_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()

Hist plots of Pitch

In [ ]:
sorted_keys = sorted(Pitch_info_extract.keys())
Pitch_info_extract = {key: Pitch_info_extract[key] for rank, key in enumerate(sorted_keys)}


nb_subplot=len(Pitch_info_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

plt.close('all')
fig,axs = plt.subplots(rows, cols, figsize=(8, 8))
axs = axs.flatten()
plt.tight_layout()

for nsubplot, nr in enumerate(Pitch_info_extract.keys()):

    # Compute the observed PSTH
    observed_psth = Pitch_info_extract[nr]
    shuffled_psths = arr = list(ShuffledPitch_info[nr].values())
    time_bins = np.linspace(-180, 180, int(360//bin_size))
    shift = int(len(time_bins)/2)  
    observed_psth = np.roll(observed_psth, shift)
    shuffled_psths = np.roll(shuffled_psths, shift)

    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_bins01 =  (observed_psth > upper_bound)

    ax = axs[nsubplot]
    
    ax.axvline(x=0, linestyle='--', color='black', alpha=0.5)
    #ax.bar(time_bins, shuffled_psths[0], color='gray', width=bin_size, alpha=0.5)
    ax.bar(time_bins, np.mean(shuffled_psths, axis=0), color='gray', width=bin_size, alpha=0.5)
    ax.bar(time_bins, observed_psth, color='orange', width=bin_size, alpha=0.5)

    ax.scatter(
        time_bins[significant_bins01],
        observed_psth[significant_bins01],
        color='black',
        label='p<0.01', s=10, marker = '*',
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim([-60, 60])
    clr= 'black' if nr in PitchCellList else 'grey'
    ftw = 'bold' if nr in PitchCellList else 'normal'
    ax.set_title(f'{nr}', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)

fig.subplots_adjust(wspace=.1, hspace=.2)
plt.savefig(f'{folder_to_save}/Pitch_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()

Hist plots of Roll

In [ ]:
sorted_keys = sorted(Roll_info_extract.keys())
Roll_info_extract = {key: Roll_info_extract[key] for rank, key in enumerate(sorted_keys)}


nb_subplot=len(Roll_info_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

plt.close('all')
fig,axs = plt.subplots(rows, cols, figsize=(8, 8))
axs = axs.flatten()
plt.tight_layout()

for nsubplot, nr in enumerate(Roll_info_extract.keys()):

    # Compute the observed PSTH
    observed_psth = Roll_info_extract[nr]
    shuffled_psths = arr = list(ShuffledRoll_info[nr].values())
    time_bins = np.linspace(-180, 180, int(360//bin_size))
    shift = int(len(time_bins)/2)  
    observed_psth = np.roll(observed_psth, shift)
    shuffled_psths = np.roll(shuffled_psths, shift)

    alpha = 0.01
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_bins01 =  (observed_psth > upper_bound)

    ax = axs[nsubplot]
    
    ax.axvline(x=0, linestyle='--', color='black', alpha=0.5)
    #ax.bar(time_bins, shuffled_psths[0], color='gray', width=bin_size, alpha=0.5)
    ax.bar(time_bins, np.mean(shuffled_psths, axis=0), color='gray', width=bin_size, alpha=0.5)
    ax.bar(time_bins, observed_psth, color='purple', width=bin_size, alpha=0.5)

    ax.scatter(
        time_bins[significant_bins01],
        observed_psth[significant_bins01],
        color='black',
        label='p<0.01', s=10, marker = '*',
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim([-90, 90])
    clr= 'black' if nr in RollCellList else 'grey'
    ftw = 'bold' if nr in RollCellList else 'normal'
    ax.set_title(f'{nr}', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)

fig.subplots_adjust(wspace=.1, hspace=.2)
plt.savefig(f'{folder_to_save}/Roll_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()

Hist plots of AHV

In [ ]:
sorted_keys = sorted(AHV_info_extract.keys())
AHV_info_extract = {key: AHV_info_extract[key] for rank, key in enumerate(sorted_keys)}


nb_subplot=len(AHV_info_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

fig, axs = plt.subplots(rows, cols, figsize=(8, 8))
axs = axs.flatten()
plt.tight_layout()

for nsubplot, nr in enumerate(AHV_info_extract.keys()):

    # Compute the observed PSTH
    observed_psth = AHV_info_extract[nr]
    shuffled_psths = arr = list(ShuffledAHV_info[nr].values())
    time_bins = np.linspace(-360, 360, int(360*2//bin_size))
    shift = int(len(time_bins)/2)  
    observed_psth = np.roll(observed_psth, shift)
    shuffled_psths = np.roll(shuffled_psths, shift)

    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_bins01 =  (observed_psth > upper_bound)
    ax = axs[nsubplot]
    
    ax.axvline(x=0, linestyle='--', color='black', alpha=0.5)
    #ax.bar(time_bins, shuffled_psths[0], color='gray', width=bin_size*2, alpha=0.5)
    ax.bar(time_bins, np.mean(shuffled_psths, axis=0), color='gray', width=bin_size*2, alpha=0.5)
    ax.bar(time_bins, observed_psth, color='blue', width=bin_size*2, alpha=0.5)

    ax.scatter(
        time_bins[significant_bins01],
        observed_psth[significant_bins01],
        color='black',
        label='p<0.01', s=10, marker = '*',
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim([-90, 90])
    clr= 'black' if nr in AHVCellList else 'grey'
    ftw = 'bold' if nr in AHVCellList else 'normal'
    ax.set_title(f'{nr}', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)

fig.subplots_adjust(wspace=.1, hspace=.2)
plt.savefig(f'{folder_to_save}/AHV_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()

Hist plots of Speed

In [ ]:
sorted_keys = sorted(Speed_info_extract.keys())
Speed_info_extract = {key: Speed_info_extract[key] for rank, key in enumerate(sorted_keys)}


nb_subplot=len(Speed_info_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

plt.close('all')
fig,axs = plt.subplots(rows, cols, figsize=(8, 8))
axs = axs.flatten()
plt.tight_layout()

for nsubplot, nr in enumerate(Speed_info_extract.keys()):

    # Compute the observed PSTH
    observed_psth = Speed_info_extract[nr]
    shuffled_psths = arr = list(ShuffledSpeed_info[nr].values())
    time_bins = np.linspace(0, 20, 20)

    alpha = 0.01
    lower_bound = np.percentile(shuffled_psths, alpha / 2 * 100, axis=0)
    upper_bound = np.percentile(shuffled_psths, (1 - alpha / 2) * 100, axis=0)
    significant_bins01 =  (observed_psth > upper_bound)

    ax = axs[nsubplot]
    
    #ax.bar(time_bins, shuffled_psths[0], color='gray', width=bin_size, alpha=0.5)
    ax.bar(time_bins, np.mean(shuffled_psths, axis=0), color='gray', width=1, alpha=0.5)
    ax.bar(time_bins, observed_psth, color='green', width=1, alpha=0.5)

    ax.scatter(
        time_bins[significant_bins01],
        observed_psth[significant_bins01],
        color='black',
        label='p<0.01', s=10, marker = '*',
    )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim([0, 10])
    clr= 'black' if nr in SpeedCellList else 'grey'
    ftw = 'bold' if nr in SpeedCellList else 'normal'
    ax.set_title(f'{nr}', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)

fig.subplots_adjust(wspace=.1, hspace=.2)
plt.savefig(f'{folder_to_save}/Speed_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()

## Per sessions

Map of place field per session

In [ ]:
sorted_keys = sorted(PlaceCell_infoSess_extract.keys())
PlaceCell_infoSess_extract = {key: PlaceCell_infoSess_extract[key] for rank, key in enumerate(sorted_keys)}

nb_subplot=len(PlaceCell_infoSess_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

fig, axs = plt.subplots(rows, cols, figsize=(10, 10))
axs = axs.flatten()
plt.tight_layout()

sigma = 1  # Standard deviation for Gaussian kernel
max_row = int(table_radius*2/square_size)+1
max_col = int(table_radius*2/square_size)+1

for nsubplot, nrsess in enumerate(PlaceCell_infoSess_extract.keys()):

    # Mask outside circle 
    rows, cols = np.full((max_row, max_col),1).shape
    center_row, center_col = rows // 2, cols // 2
    radius = min(rows, cols) // 2   
    Y, X = np.ogrid[:rows, :cols]# Create a circular mask
    dist_from_center = np.sqrt((X - center_col)**2 + (Y - center_row)**2)
    mask = dist_from_center < radius    
    masked_array = np.where(mask, np.full((max_row, max_col),1), np.nan)# Apply mask: set values outside circle to NaN
    
    # Draw map
    axs[nsubplot].imshow(masked_array, cmap='Greys_r', origin='upper')
    axs[nsubplot].imshow(PlaceCell_infoSess_extract[nrsess], cmap='jet', origin='upper')
    axs[nsubplot].set_aspect('equal')
    nr2='_'.join(nrsess.split('_')[:2])
    nr=nrsess.split('_')[0]
    clr= 'black' if nr in PlaceCellList else 'grey'
    ftw = 'bold' if nr in PlaceCellList else 'normal'
    axs[nsubplot].set_title(f'{nr2}', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)
    #axs[nsubplot].set_title(f'{nr2} ({np.round(Spatial_info[nr], 1)})', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)
        
    # Remove box (spines)
    for spine in axs[nsubplot].spines.values():
        spine.set_visible(False)
    axs[nsubplot].set_xticks([])  # No x-axis ticks
    axs[nsubplot].set_yticks([])  # No y-axis ticks

# Adjust layout to avoid clipping
fig.subplots_adjust(wspace=.1, hspace=.2)
plt.savefig(f'{folder_to_save}/PlaceFields_perSess_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()


Polar plot of HD per session

In [ ]:
sorted_keys = sorted(HeadDirection_infoSess_extract.keys())
HeadDirection_infoSess_extract = {key: HeadDirection_infoSess_extract[key] for rank, key in enumerate(sorted_keys)}

nb_subplot=len(HeadDirection_infoSess_extract.keys())
rows = int(np.ceil(np.sqrt(nb_subplot)))  # Rows: ceil(sqrt(X))
cols = int(np.ceil(np.sqrt(nb_subplot)))  # Columns: floor(sqrt(X))

# Create the figure with a 2x2 grid
fig, axs = plt.subplots(rows, cols, figsize=(10, 10),  subplot_kw={'projection': 'polar'})
axs = axs.flatten()
plt.tight_layout()

for nsubplot, nrsess in enumerate(HeadDirection_infoSess_extract.keys()):

    ax = axs[nsubplot]
    ax.set_theta_zero_location("E")  # 0° at East
    ax.set_theta_direction(-1)        # Clockwise
    nr2='_'.join(nrsess.split('_')[:2])
    nr=nrsess.split('_')[0]
    
    clr= 'black' if nr in HeadDirectionCellList else 'grey'
    ftw = 'bold' if nr in HeadDirectionCellList else 'normal'
    ax.set_title(f'{nr2}', color = clr, pad=0, loc='left', fontweight=ftw, fontsize=7)
    
    theta = bin_centers
    theta = np.append(theta, theta[0])  
    r_smooth = gaussian_filter1d(HeadDirection_infoSess_extract[nrsess], sigma=1, mode='wrap')  # wrap for circular smoothing
    r_smooth = np.append(r_smooth, r_smooth[0])
    pref_idx = np.argmax(r_smooth)    # Find preferred angle
    pref_angle = theta[pref_idx]    
    color = plt.cm.hsv(pref_angle / (2*np.pi))# pref_angle ranges 0 to 2*pi -> normalize to 0-1 for colormap
    ax.plot(theta, r_smooth, color=color)
    ax.set_xticks([])
    ax.set_yticks([])

fig.subplots_adjust(wspace=.1, hspace=.1)
plt.savefig(f'{folder_to_save}/HeadDirection_perSess_extract{rand_id}.svg', format='svg', bbox_inches='tight', pad_inches=0, transparent=True) if save else None
plt.show()
